In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

Train a car-playing RL Agent
==============================

**Authors:** [Yuansong Feng](https://github.com/YuansongFeng), [Suraj
Subramanian](https://github.com/suraj813), [Howard
Wang](https://github.com/hw26), [Steven
Guo](https://github.com/GuoYuzhang).

This tutorial walks you through the fundamentals of Deep Reinforcement
Learning. At the end, you will implement an AI-powered car (using
[Double Deep Q-Networks](https://arxiv.org/pdf/1509.06461.pdf)) that can
play the game by itself.

Although no prior knowledge of RL is necessary for this tutorial, you
can familiarize yourself with these RL
[concepts](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html),
and have this handy
[cheatsheet](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N)
as your companion. The full code is available
[here](https://github.com/yuansongFeng/Madcar/).

![](https://pytorch.org/tutorials/_static/img/car.gif)


``` {.bash}
%%bash
pip install gym-super-car-bros==7.4.0
pip install tensordict==0.3.0
pip install torchrl==0.3.0
```


In [ ]:
import torch
from torch import nn
from torchvision import transforms as T
from PIL import Image
import numpy as np
from pathlib import Path
from collections import deque
import random, datetime, os

import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Box
# from gymnasium.wrappers import FrameStack
# from nes_py.wrappers import JoypadSpace
# import ale_py
# from ale_py import ALEInterface
# ale = ALEInterface()
from tensordict import TensorDict
from torchrl.data import TensorDictReplayBuffer, LazyMemmapStorage


RL Definitions
==============

**Environment** The world that an agent interacts with and learns from.

**Action** $a$ : How the Agent responds to the Environment. The set of
all possible Actions is called *action-space*.

**State** $s$ : The current characteristic of the Environment. The set
of all possible States the Environment can be in is called
*state-space*.

**** $r$ :  is the key feedback from Environment to Agent.
It is what drives the Agent to learn and to change its future action. An
aggregation of s over multiple time steps is called **Return**.

**Optimal Action-Value function** $Q^*(s,a)$ : Gives the expected return
if you start in state $s$, take an arbitrary action $a$, and then for
each future time step take the action that maximizes returns. $Q$ can be
said to stand for the "quality" of the action in a state. We try to
approximate this function.


Environment
===========

Initialize Environment
----------------------

In car, the environment consists of tubes, mushrooms and other
components.

When car makes an action, the environment responds with the changed
(next) state, reward and other info.


In [ ]:
# !pip install 'numpy<2.0.0'
gym.__version__

In [ ]:
import sys
import os

print(f"Current Python Version: {sys.version}")
print(f"Executable Path: {sys.executable}")

In [ ]:

env = gym.make("CarRacing-v3", render_mode="rgb_array", lap_complete_percent=0.95, domain_randomize=False, continuous=False, max_episode_steps=1000)
env.reset()
next_state, reward, done, trunc, info = env.step(action=0)
print(f"\n reward {reward},\n done {done},\n info {info}")
env.close()
print("Process finished safely.")

<!-- Preprocess Environment
======================

Environment data is returned to the agent in `next_state`. As you saw
above, each state is represented by a `[3, 240, 256]` size array. Often
that is more information than our agent needs; for instance, car's
actions do not depend on the color of the pipes or the sky!

We use **Wrappers** to preprocess environment data before sending it to
the agent.

`GrayScaleObservation` is a common wrapper to transform an RGB image to
grayscale; doing so reduces the size of the state representation without
losing useful information. Now the size of each state: `[1, 240, 256]`

`ResizeObservation` downsamples each observation into a square image.
New size: `[1, 84, 84]`

`SkipFrame` is a custom wrapper that inherits from `gym.Wrapper` and
implements the `step()` function. Because consecutive frames don't vary
much, we can skip n-intermediate frames without losing much information.
The n-th frame aggregates rewards accumulated over each skipped frame.

`FrameStack` is a wrapper that allows us to squash consecutive frames of
the environment into a single observation point to feed to our learning
model. This way, we can identify if car was landing or jumping based
on the direction of his movement in the previous several frames. -->


After applying the above wrappers to the environment, the final wrapped
state consists of 4 gray-scaled consecutive frames stacked together, as
shown above in the image on the left. Each time car makes an action,
the environment responds with a state of this structure. The structure
is represented by a 3-D array of size `[4, 84, 84]`.

![](https://pytorch.org/tutorials/_static/img/car_env.png)


Agent
=====

We create a class `car` to represent our agent in the game. car
should be able to:

-   **Act** according to the optimal action policy based on the current
    state (of the environment).
-   **Remember** experiences. Experience = (current state, current
    action, reward, next state). car *caches* and later *recalls* his
    experiences to update his action policy.
-   **Learn** a better action policy over time


In [ ]:
class Car:
    def __init__():
        pass

    def act(self, state):
        """Given a state, choose an epsilon-greedy action"""
        pass

    def cache(self, experience):
        """Add the experience to memory"""
        pass

    def recall(self):
        """Sample experiences from memory"""
        pass

    def learn(self):
        """Update online action value (Q) function with a batch of experiences"""
        pass

In the following sections, we will populate car's parameters and
define his functions.


Act
===

For any given state, an agent can choose to do the most optimal action
(**exploit**) or a random action (**explore**).

car randomly explores with a chance of `self.exploration_rate`; when
he chooses to exploit, he relies on `carNet` (implemented in `Learn`
section) to provide the most optimal action.


In [ ]:
class Car:
    def __init__(self, state_dim, input_dim, output_dir):
        self.state_dim = state_dim
        self.input_dim = input_dim
        self.output_dir = output_dir
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")

        self.net = carNet(state_dim, input_dim, output_dir).float()
        self.net = self.net.to(device=self.device)

        self.exploration_rate = 1
        self.exploration_rate_decay = 0.9995
        self.exploration_rate_min = 0.05
        self.curr_step = 0
        self.learn_step_interval = 3  
        self.burnin = 1e3             
        self.sync_every = 1e3         
        self.curr_step = 0            

        self.save_every = 1e3

    def act(self, state):
        self.curr_step += 1
        if self.curr_step > self.burnin:
            self.exploration_rate *= self.exploration_rate_decay
            self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)
        # EXPLORE
        x = np.random.rand()
        if np.random.rand() < self.exploration_rate:
            if np.random.rand() < 0.5:
                action_idx = 3
            else:
                action_idx = np.random.randint(low=1, high=self.input_dim)

        # EXPLOIT
        else:
            state = np.array(state)
            
            state = torch.tensor(state, device=self.device).float()
            state = state
            
            if state.ndim == 3:
                state = state.unsqueeze(0)
            
            # if state.shape[1] != 4:
            #     raise ValueError(f"Expected 4 channels, got {state.shape[1]}. Check FrameStack!")

            # action_values = self.net(state, model="online")
            # # if np.random.rand() < 0.5:
            # #     action_idx = 3  # Force Gas
            # # else:
            # #     # action_idx = np.random.randint(self.input_dim)
            # action_idx = torch.argmax(action_values, axis=1).item()

            # # decrease exploration_rate
            # self.exploration_rate *= self.exploration_rate_decay
            # self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)
            # self.net.eval()
            with torch.no_grad():
                action_values = self.net(state, model="online")
            action_idx = torch.argmax(action_values, axis=1).item()

        return action_idx

Cache and Recall
================

These two functions serve as car's "memory" process.

`cache()`: Each time car performs an action, he stores the
`experience` to his memory. His experience includes the current *state*,
*action* performed, *reward* from the action, the *next state*, and
whether the game is *done*.

`recall()`: car randomly samples a batch of experiences from his
memory, and uses that to learn the game.


In [ ]:
class Car(Car):  # subclassing for continuity
    def __init__(self, state_dim, input_dim, output_dir):
        super().__init__(state_dim, input_dim, output_dir)
        self.memory = TensorDictReplayBuffer(storage=LazyMemmapStorage(100000, device=torch.device("cpu")))
        self.batch_size = 64

    def cache(self, state, next_state, action, reward, done):
        """
        Store the experience to self.memory (replay buffer)

        Inputs:
        state (``LazyFrame``),
        next_state (``LazyFrame``),
        action (``int``),
        reward (``float``),
        done(``bool``))
        """
        def first_if_tuple(x):
            return x[0] if isinstance(x, tuple) else x
        state = first_if_tuple(state).__array__()
        next_state = first_if_tuple(next_state).__array__()

        state = torch.tensor(state)
        next_state = torch.tensor(next_state)
        action = torch.tensor([action])
        reward = torch.tensor([reward])
        done = torch.tensor([done])

        self.memory.add(TensorDict({"state": state, "next_state": next_state, "action": action, "reward": reward, "done": done}, batch_size=[]))

    def recall(self):
        """
        Retrieve a batch of experiences from memory
        """
        batch = self.memory.sample(self.batch_size).to(self.device)
        state, next_state, action, reward, done = (batch.get(key) for key in ("state", "next_state", "action", "reward", "done"))
        return state, next_state, action.squeeze(), reward.squeeze(), done.squeeze()
    def learn(self):

        state, next_state, action, reward, done = self.recall()
        state = state.float().squeeze(-1) 
        next_state = next_state.float().squeeze(-1) 

        td_est = self.td_estimate(state, action)

        td_tgt = self.td_target(next_state, reward, done)
        loss = self.update_Q_online(td_est, td_tgt)

        if self.curr_step % self.sync_every == 0:
            self.sync_Q_target()

        return loss

Learn
=====

car uses the [DDQN algorithm](https://arxiv.org/pdf/1509.06461) under
the hood. DDQN uses two ConvNets - $Q_{online}$ and $Q_{target}$ - that
independently approximate the optimal action-value function.

In our implementation, we share feature generator `features` across
$Q_{online}$ and $Q_{target}$, but maintain separate FC classifiers for
each. $\theta_{target}$ (the parameters of $Q_{target}$) is frozen to
prevent updating by backprop. Instead, it is periodically synced with
$\theta_{online}$ (more on this later).

Neural Network
--------------


In [ ]:
class carNet(nn.Module):
    """mini CNN structure
  input -> (conv2d + relu) x 3 -> flatten -> (dense + relu) x 2 -> output
  """

    def __init__(self, state_dim, input_dim, output_dir):
        super().__init__()
        c, h, w = state_dim

        # if h != 84:
        #     raise ValueError(f"Expecting input height: 84, got: {h}")
        # if w != 84:
        #     raise ValueError(f"Expecting input width: 84, got: {w}")

        self.online = self.__build_cnn(c, input_dim)

        self.target = self.__build_cnn(c, input_dim)
        self.target.load_state_dict(self.online.state_dict())

        # Q_target parameters are frozen.
        for p in self.target.parameters():
            p.requires_grad = False

    def forward(self, input, model):
        if model == "online":
            return self.online(input)
        elif model == "target":
            return self.target(input)

    def __build_cnn(self, c, input_dim):
        return nn.Sequential(
            nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
        )

TD Estimate & TD Target
=======================

Two values are involved in learning:

**TD Estimate** - the predicted optimal $Q^*$ for a given state $s$

$${TD}_e = Q_{online}^*(s,a)$$

**TD Target** - aggregation of current reward and the estimated $Q^*$ in
the next state $s'$

$$a' = argmax_{a} Q_{online}(s', a)$$

$${TD}_t = r + \gamma Q_{target}^*(s',a')$$

Because we don't know what next action $a'$ will be, we use the action
$a'$ maximizes $Q_{online}$ in the next state $s'$.

Notice we use the
[\@torch.no\_grad()](https://pytorch.org/docs/stable/generated/torch.no_grad.html#no-grad)
decorator on `td_target()` to disable gradient calculations here
(because we don't need to backpropagate on $\theta_{target}$).


In [ ]:
class Car(Car):
    def __init__(self, state_dim, input_dim, output_dir):
        super().__init__(state_dim, input_dim, output_dir)
        self.gamma = 0.95

    def td_estimate(self, state, action):
        if state.dtype == torch.uint8:
            state = state.float()
        
        if state.ndimension() == 5:
            state = state.squeeze(-1)

        current_Q = self.net(state, model="online")[
            np.arange(0, self.batch_size), action
        ]
        return current_Q

    @torch.no_grad()
    def td_target(self, next_state, reward, done):
        next_state = next_state.to(device=self.device, dtype=torch.float32)
        
        if next_state.ndimension() == 5:
            next_state = next_state.squeeze(-1)
        
        next_state = next_state 

        next_state_Q = self.net(next_state, model="online")
        best_action = torch.argmax(next_state_Q, axis=1)

        next_Q = self.net(next_state, model="target")[
            np.arange(0, self.batch_size), best_action
        ]
        
        # Ensure reward and done are also on the same device
        reward = reward.to(self.device)
        done = done.to(self.device)

        return (reward + (1 - done.float()) * self.gamma * next_Q).float()

Updating the model
==================

As car samples inputs from his replay buffer, we compute $TD_t$ and
$TD_e$ and backpropagate this loss down $Q_{online}$ to update its
parameters $\theta_{online}$ ($\alpha$ is the learning rate `lr` passed
to the `optimizer`)

$$\theta_{online} \leftarrow \theta_{online} + \alpha \nabla(TD_e - TD_t)$$

$\theta_{target}$ does not update through backpropagation. Instead, we
periodically copy $\theta_{online}$ to $\theta_{target}$

$$\theta_{target} \leftarrow \theta_{online}$$


In [ ]:
class Car(Car):
    def __init__(self, state_dim, input_dim, output_dir):
        super().__init__(state_dim, input_dim, output_dir)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
        self.loss_fn = torch.nn.SmoothL1Loss()

    def update_Q_online(self, td_estimate, td_target):
        loss = self.loss_fn(td_estimate, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def sync_Q_target(self):
        self.net.target.load_state_dict(self.net.online.state_dict())

Save checkpoint
===============


In [ ]:
class Car(Car):
    def save(self):
        save_path = (
            f"car_net_short.chkpt"
        )
        torch.save(
            dict(model=self.net.state_dict(), exploration_rate=self.exploration_rate),
            save_path,
        )
        print(f"carNet saved to {save_path} at step {self.curr_step}")

Putting it all together
=======================


In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal) backend!")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU.")

In [ ]:
class Car(Car):
    def __init__(self, state_dim, input_dim, output_dim):
        super().__init__(state_dim, input_dim, output_dim)
        self.device = device
        self.net = carNet(state_dim, input_dim, output_dim).to(self.device)
        self.burnin = 1e3  # min. experiences before training
        self.learn_every = 3  # no. of experiences between updates to Q_online
        self.sync_every = 1e3  # no. of experiences between Q_target & Q_online sync

    def learn(self):
        if self.curr_step % self.sync_every == 0:
            self.sync_Q_target()

        # if self.curr_step % self.save_every == 0:
        #     self.save()

        if self.curr_step < self.burnin:
            return None, None

        if self.curr_step % self.learn_every != 0:
            return None, None

        state, next_state, action, reward, done = self.recall()

        td_est = self.td_estimate(state, action)

        td_tgt = self.td_target(next_state, reward, done)

        # Backpropagate loss through Q_online
        loss = self.update_Q_online(td_est, td_tgt)

        return (td_est.mean().item(), loss)

Logging
=======


In [ ]:
import numpy as np
import time, datetime
import matplotlib.pyplot as plt


class MetricLogger:
    def __init__(self, output_dir):
        self.save_log = output_dir / "log"
        with open(self.save_log, "w") as f:
            f.write(
                f"{'Episode':>8}{'Step':>8}{'Epsilon':>10}{'MeanReward':>15}"
                f"{'MeanLength':>15}{'MeanLoss':>15}{'MeanQValue':>15}"
                f"{'TimeDelta':>15}{'Time':>20}\n"
            )
        self.ep_rewards_plot = output_dir / "reward_plot.jpg"
        self.ep_lengths_plot = output_dir / "length_plot.jpg"
        self.ep_avg_losses_plot = output_dir / "loss_plot.jpg"
        self.ep_avg_qs_plot = output_dir / "q_plot.jpg"

        # History metrics
        self.ep_rewards = []
        self.ep_lengths = []
        self.ep_avg_losses = []
        self.ep_avg_qs = []

        # Moving averages, added for every call to record()
        self.moving_avg_ep_rewards = []
        self.moving_avg_ep_lengths = []
        self.moving_avg_ep_avg_losses = []
        self.moving_avg_ep_avg_qs = []

        # Current episode metric
        self.init_episode()

        # Timing
        self.record_time = time.time()

    def log_step(self, reward, loss, q):
        self.curr_ep_reward += reward
        self.curr_ep_length += 1
        if loss:
            self.curr_ep_loss += loss
            self.curr_ep_q += q
            self.curr_ep_loss_length += 1

    def log_episode(self):
        "Mark end of episode"
        self.ep_rewards.append(self.curr_ep_reward)
        self.ep_lengths.append(self.curr_ep_length)
        if self.curr_ep_loss_length == 0:
            ep_avg_loss = 0
            ep_avg_q = 0
        else:
            ep_avg_loss = np.round(self.curr_ep_loss / self.curr_ep_loss_length, 5)
            ep_avg_q = np.round(self.curr_ep_q / self.curr_ep_loss_length, 5)
        self.ep_avg_losses.append(ep_avg_loss)
        self.ep_avg_qs.append(ep_avg_q)

        self.init_episode()

    def init_episode(self):
        self.curr_ep_reward = 0.0
        self.curr_ep_length = 0
        self.curr_ep_loss = 0.0
        self.curr_ep_q = 0.0
        self.curr_ep_loss_length = 0

    def record(self, episode, epsilon, step):
        mean_ep_reward = np.round(np.mean(self.ep_rewards[-100:]), 3)
        mean_ep_length = np.round(np.mean(self.ep_lengths[-100:]), 3)
        mean_ep_loss = np.round(np.mean(self.ep_avg_losses[-100:]), 3)
        mean_ep_q = np.round(np.mean(self.ep_avg_qs[-100:]), 3)
        self.moving_avg_ep_rewards.append(mean_ep_reward)
        self.moving_avg_ep_lengths.append(mean_ep_length)
        self.moving_avg_ep_avg_losses.append(mean_ep_loss)
        self.moving_avg_ep_avg_qs.append(mean_ep_q)

        last_record_time = self.record_time
        self.record_time = time.time()
        time_since_last_record = np.round(self.record_time - last_record_time, 3)

        print(
            f"Episode {episode} - "
            f"Step {step} - "
            f"Epsilon {epsilon} - "
            f"Mean Reward {mean_ep_reward} - "
            f"Mean Length {mean_ep_length} - "
            f"Mean Loss {mean_ep_loss} - "
            f"Mean Q Value {mean_ep_q} - "
            f"Time Delta {time_since_last_record} - "
            f"Time {datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S')}"
        )

        with open(self.save_log, "a") as f:
            f.write(
                f"{episode:8d}{step:8d}{epsilon:10.3f}"
                f"{mean_ep_reward:15.3f}{mean_ep_length:15.3f}{mean_ep_loss:15.3f}{mean_ep_q:15.3f}"
                f"{time_since_last_record:15.3f}"
                f"{datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S'):>20}\n"
            )

        for metric in ["ep_lengths", "ep_avg_losses", "ep_avg_qs", "ep_rewards"]:
            plt.clf()
            plt.plot(getattr(self, f"moving_avg_{metric}"), label=f"moving_avg_{metric}")
            plt.legend()
            plt.savefig(getattr(self, f"{metric}_plot"))

Let's play!
===========

In this example we run the training loop for 40 episodes, but for car
to truly learn the ways of his world, we suggest running the loop for at
least 40,000 episodes!


Conclusion
==========

In this tutorial, we saw how we can use PyTorch to train a game-playing
AI. You can use the same methods to train an AI to play any of the games
at the [OpenAI gym](https://gym.openai.com/). Hope you enjoyed this
tutorial, feel free to reach us at [our
github](https://github.com/yuansongFeng/Madcar/)!


In [ ]:
import logging
training_period = 20          # Record video every 250 episodes
num_training_episodes = 61  # Total training episodes
# env_name = "car"

logging.basicConfig(level=logging.INFO, format='%(message)s')

In [ ]:
env

In [ ]:
# import gymnasium as gym
# from gymnasium.wrappers.transform_observation import TransformObservation, GrayscaleObservation, ResizeObservation
# from gymnasium.wrappers.stateful_observation import FrameStackObservation as FrameStack
# from gymnasium.wrappers import RecordVideo
# from shapely import affinity
# from shapely.geometry import Point, Polygon
# from tqdm import tqdm
# pbar = tqdm(total=num_training_episodes, desc="Training car")
# class SafeDrivingWrapper(gym.Wrapper): #https://medium.com/@gomes.fs/ai-car-racing-teach-ai-to-drive-a2049ee07297
#     def __init__(self, env, border_width=0.7):
#         super(SafeDrivingWrapper, self).__init__(env)
#         self.border_width = border_width

#     def car_on_track(self):
#         car_on_track = False
#         x, y = self.unwrapped.car.hull.position
#         point = Point(x, y)
#         for poly in self.unwrapped.road_poly:
#             polygon = Polygon(poly[0])

#             if self.border_width > 0:
#                 border_scale = 1 + self.border_width
#                 polygon = affinity.scale(polygon, xfact=border_scale, yfact=border_scale)
#             if polygon.contains(point):
#                 car_on_track = True
#                 break
#         return car_on_track

#     def step(self, action):
#         next_state, reward, terminated, truncated, info = self.env.step(action)

#         if not self.car_on_track():
#             reward -= 100
#             terminated = True 

#         return next_state, reward, terminated, truncated, info
# class RewardSpeed(gym.Wrapper):
#     def get_car_speed(self):
#         vel = self.env.unwrapped.car.hull.linearVelocity
#         return np.sqrt(vel[0]**2 + vel[1]**2)
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         speed_bonus = self.get_car_speed() * 1
#         info['speed'] = self.get_car_speed()
#         total_reward = reward + speed_bonus

#         return obs, total_reward, terminated, truncated, info

# env = GrayscaleObservation(env, keep_dim=False)
# env = RewardSpeed(env)
# env = SafeDrivingWrapper(env)
# env = ResizeObservation(env, (84, 84))
# new_space = gym.spaces.Box(low=0, high=255, shape=(1, 84, 84), dtype=np.uint8)
# env = TransformObservation(env, lambda obs: obs.squeeze() if obs.ndim > 2 else obs, observation_space=env.observation_space)
# env = FrameStack(env, 4) 
# env = RecordVideo(env, video_folder="track-short", episode_trigger=lambda x: x % training_period == 0)
# car = Car(state_dim=env.observation_space.shape, input_dim=env.action_space.n, output_dim=env.action_space.n)
# all_rewards = []
# speed = []
# for episode_num in range(num_training_episodes):
#     state, info = env.reset()

#     episode_over = False
#     episode_reward = 0
#     episode_steps = 0
#     episode_speeds = []
#     slow_steps = 0
    
#     while not episode_over:
#         action = car.act(state)

#         next_state, reward, terminated, truncated, info = env.step(action)
#         reward = round(reward)
#         current_speed = info.get('speed', 0)
#         episode_speeds.append(current_speed)
#         if info['speed'] < 5.0:
#             reward -= abs(5 - info['speed'])
#             if info['speed'] < 2.5:
#                 slow_steps +=1
#         if slow_steps > 50:
#             terminated = True
#         progress_reward = env.unwrapped.tile_visited_count / len(env.unwrapped.track)
#         reward = reward + (progress_reward * 10)
#         reward = np.float32(reward)
#         if reward < -50:
#             terminated = True
#         done = terminated or truncated
        
#         car.cache(state, next_state, action, reward, done)
#         loss = car.learn() 

#         state = next_state
#         episode_reward += reward
#         episode_steps += 1
        
#         if done or info.get("flag_get", False):
#             all_rewards.append(episode_reward)
#             # print('speed: ', info['speed'])
#             break
#         episode_over = done

#     if "episode" in info:
#         episode_data = info["episode"]
#         logging.info(f"Episode {episode_num}: "
#                     f"reward={episode_data['r']:.1f}, "
#                     f"length={episode_data['l']}, "
#                     f"time={episode_data['t']:.2f}s")

#     avg_speed = sum(episode_speeds) / len(episode_speeds) if episode_speeds else 0
#     if episode_num % 10 == 0:
#         recent_rewards = all_rewards[:-11:-1]
#         avg_recent = sum(recent_rewards) / len(recent_rewards)
#         print(f"  -> Average reward over last {len(recent_rewards)} episodes: {avg_recent:.1f}, speed: {avg_speed}")
#     pbar.set_postfix({
#         "Reward": f"{episode_reward:.1f}",
#         "Epsilon": f"{car.exploration_rate:.2f}",
#         "Step": car.curr_step,
#         "Speed": info['speed']
#     })
#     pbar.update(1)
#     if episode_num == num_training_episodes -1:
#         car.save()
    
# pbar.close()
# env.close()


## bigger training

In [1]:
import torch
from torch import nn
from torchvision import transforms as T
from PIL import Image
import numpy as np
from pathlib import Path
from collections import deque
import random, datetime, os

import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Box
from tensordict import TensorDict
from torchrl.data import TensorDictReplayBuffer, LazyMemmapStorage


/Users/iamgeorgerieh/Documents/coding/.venv/lib/python3.10/site-packages/torchrl/data/replay_buffers/samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)


In [2]:

env = gym.make("CarRacing-v3", render_mode="rgb_array", lap_complete_percent=0.95, domain_randomize=False, continuous=False, max_episode_steps=5000)
env.reset()
next_state, reward, done, trunc, info = env.step(action=0)
print(f"\n reward {reward},\n done {done},\n info {info}")
env.close()
print("Process finished safely.")

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
/Users/iamgeorgerieh/Documents/coding/.venv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists



 reward 6.500660066006601,
 done False,
 info {'speed': 9.99766867613486e-07}
Process finished safely.


In [3]:
class Car:
    def __init__():
        pass

    def act(self, state):
        """Given a state, choose an epsilon-greedy action"""
        pass

    def cache(self, experience):
        """Add the experience to memory"""
        pass

    def recall(self):
        """Sample experiences from memory"""
        pass

    def learn(self):
        """Update online action value (Q) function with a batch of experiences"""
        pass

In [4]:
class Car:
    
    def __init__(self, state_dim, input_dim, output_dim):
        self.state_dim = state_dim
        self.input_dim = input_dim
        self.output_dim = output_dim
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")

        total_in_channels = state_dim[0]

        self.net = carNet(state_dim, input_dim, total_in_channels).float().to(self.device)
        self.gamma = 0.9999
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
        self.loss_fn = torch.nn.SmoothL1Loss()

        self.memory = TensorDictReplayBuffer(storage=LazyMemmapStorage(100000, device=torch.device("cpu")))
        self.batch_size = 64

        self.exploration_rate = 1
        self.exploration_rate_decay = 0.9999
        self.exploration_rate_min = 0.05
        self.curr_step = 0
        self.learn_step_interval = 4 
        self.burnin = 1e3          
        self.sync_every = 1e3     
        self.learn_every = 4     
        self.save_every = 1e3
        self.curr_step = 0 
        self._skip_frames = 4  
        self.output_dir = './saved_models'         


    def act(self, state, speed):
        self.curr_step += 1
        
        # 1. EXPLORE PHASE
        x = np.random.rand()
        if x < self.exploration_rate:
            # Decay epsilon only after we cross burnin
            if self.curr_step > self.burnin:
                self.exploration_rate = max(
                    self.exploration_rate_min, 
                    self.exploration_rate * self.exploration_rate_decay
                )
                
            # Rule-based exploration is safe here because the buffer 
            # will log that a random action was forced.
            if speed < 30:
                return 3  # Assuming 3 is your acceleration/gas action
            elif speed < 50:
                return np.random.randint(low=1, high=4)
            else:
                return np.random.randint(low=0, high=self.input_dim)
                    
        # 2. EXPLOIT PHASE
        else:
            state = np.array(state)
            state = torch.tensor(state, device=self.device).float()
            
            if state.ndim == 3:
                state = state.unsqueeze(0)
            
            self.net.eval()
            with torch.no_grad():
                action_values = self.net(state, model="online")
            
            # SAFE OVERRIDE: Modify the network values directly before argmax
            # if speed < 30:
            #     # Force the network to choose action 3 by making its score infinitely high
            #     forced_actions = torch.zeros_like(action_values)
            #     forced_actions[0, 3] = 99999.0
            #     return torch.argmax(forced_actions, axis=1).item()
            # else:
                return torch.argmax(action_values, axis=1).item()

In [5]:

class Car(Car):  # subclassing for continuity
    def __init__(self, state_dim, input_dim, output_dim):
        super().__init__(state_dim, input_dim, output_dim)


    def cache(self, state, next_state, action, reward, done):
        """Saves data keeping (Frames, H, W) layout intact."""
        def preprocess(s):
           return torch.tensor(np.array(s), dtype=torch.float32)

        state = preprocess(state)
        next_state = preprocess(next_state)
        
        action = torch.tensor([action], dtype=torch.long)
        reward = torch.tensor([reward], dtype=torch.float32)
        done = torch.tensor([done], dtype=torch.bool)

        self.memory.add(TensorDict({
            "state": state, 
            "next_state": next_state, 
            "action": action, 
            "reward": reward, 
            "done": done
        }, batch_size=[]))

    def recall(self):
        """Retrieves a coherent mini-batch of experiences from the replay buffer."""
        batch = self.memory.sample(self.batch_size).to(self.device)
        
        state = batch.get("state")
        next_state = batch.get("next_state")
        action = batch.get("action")
        reward = batch.get("reward")
        done = batch.get("done")
        
        return state, next_state, action.squeeze(-1), reward.squeeze(-1), done.squeeze(-1)

    def learn(self):
        if self.curr_step % self.sync_every == 0:
            self.sync_Q_target()

        if self.curr_step % (self.save_every * 10) == 0:
            self.save()

        if self.curr_step < self.burnin:
            return None, None

        if self.curr_step % self.learn_every != 0:
            return None, None

        state, next_state, action, reward, done = self.recall()
        state = state.to(device=self.device, dtype=torch.float32) / 255.0
        next_state = next_state.to(device=self.device, dtype=torch.float32) / 255.0

        td_est = self.td_estimate(state, action)
        td_tgt = self.td_target(next_state, reward, done)

        loss = self.update_Q_online(td_est, td_tgt)

        return (td_est.mean().item(), loss)

    def save(self, suffix=""):
        os.makedirs(self.output_dir, exist_ok=True)
        filename = f"car_net_{suffix}.pt" if suffix else "car_net.pt"
        save_path = os.path.join(self.output_dir, filename)
        torch.save({
            'online': self.net.state_dict(),
            'exploration_rate': self.exploration_rate,
            'curr_step': self.curr_step
        }, save_path)

    def load(self, suffix="best"):
        filename = f"car_net_{suffix}.pt" if suffix else "car_net.pt"
        load_path = os.path.join(self.output_dir, filename)
        if not os.path.exists(load_path):
            return False
            
        checkpoint = torch.load(load_path, map_location=self.device)
        state_dict = checkpoint['online']
        
        first_key = next(iter(state_dict))
        if first_key.startswith("online."):
            from collections import OrderedDict
            clean_state_dict = OrderedDict()
            for k, v in state_dict.items():
                if k.startswith("online."):
                    clean_state_dict[k.replace("online.", "")] = v
            state_dict = clean_state_dict
        self.net.online.load_state_dict(state_dict)
        self.net.target.load_state_dict(state_dict)
        
        self.exploration_rate = checkpoint.get('exploration_rate', self.exploration_rate)
        self.curr_step = checkpoint.get('curr_step', self.curr_step)
        return True

In [6]:
class carNet(nn.Module):
    """mini CNN structure
  input -> (conv2d + relu) x 3 -> flatten -> (dense + relu) x 2 -> output
  """

    def __init__(self, state_dim, input_dim, output_dim):
        super().__init__()
        # print(state_dim)
        # f, h, w, c = state_dim
        # total_in_channels = f * c
        h, w, c = state_dim
        total_in_channels = output_dim

        self.online = self.__build_cnn(total_in_channels, input_dim)

        self.target = self.__build_cnn(total_in_channels, input_dim)
        self.target.load_state_dict(self.online.state_dict())

        # Q_target parameters are frozen.
        for p in self.target.parameters():
            p.requires_grad = False

    def forward(self, input, model):
        if model == "online":
            return self.online(input)
        elif model == "target":
            return self.target(input)

    def __build_cnn(self, c, input_dim):
        return nn.Sequential(

            nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            
        )

In [7]:
class Car(Car):
    def __init__(self, state_dim, input_dim, output_dim):
        super().__init__(state_dim, input_dim, output_dim)

    def td_estimate(self, state, action):
        current_Q = self.net(state, model="online")[
            torch.arange(0, self.batch_size, device=self.device), action
        ]
        return current_Q

    def td_target(self, next_state, reward, done):
        reward = reward.to(self.device)
        done = done.to(self.device)
        
        with torch.no_grad():
            next_state_Q = self.net(next_state, model="online")
            best_action = torch.argmax(next_state_Q, axis=1)

            next_Q = self.net(next_state, model="target")[
                torch.arange(0, self.batch_size, device=self.device), best_action
            ]
        
        return reward + (1 - done.float()) * self.gamma * next_Q

In [ ]:
class Car(Car):
    def __init__(self, state_dim, input_dim, output_dim):
        super().__init__(state_dim, input_dim, output_dim)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
        self.loss_fn = torch.nn.SmoothL1Loss()

    def update_Q_online(self, td_estimate, td_target):
        loss = self.loss_fn(td_estimate, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()
    # If the car got an unexpectedly high reward (e.g., hitting a new track tile), the loss forces the network to increase the Q-value for that specific steer/gas action in that visual state. Next time it sees that corner, it will be more likely to choose that action.If the car unexpectedly crashed (e.g., got a $-300$ penalty), the loss forces the network to brutally crush the Q-value for whatever action led to it.

    def sync_Q_target(self):
        self.net.target.load_state_dict(self.net.online.state_dict())

In [9]:
import logging
training_period = 25         # Record video every 50 episodes
num_training_episodes = 2000  # Total training episodes

logging.basicConfig(level=logging.INFO, format='%(message)s')

In [10]:
import gymnasium as gym
from gymnasium.wrappers.transform_observation import TransformObservation, GrayscaleObservation, ResizeObservation
from gymnasium.wrappers.stateful_observation import FrameStackObservation as FrameStack
from gymnasium.wrappers import RecordVideo
from shapely import affinity
from shapely.geometry import Point, Polygon
from tqdm import tqdm
import math
pbar = tqdm(total=num_training_episodes, desc="Training car")

class CarEnvironment(gym.Wrapper):
    def __init__(self, env, skip_frames=2, no_operation=0, **kwargs):
        super().__init__(env, **kwargs)
        self._no_operation = no_operation
        self._skip_frames = skip_frames

    def reset(self, *, seed=None, options=None):
        observation, info = self.env.reset(seed=seed, options=options)
        
        for _ in range(self._no_operation):
            observation, reward, terminated, truncated, info = self.env.step(0)
            if terminated or truncated:
                observation, info = self.env.reset()
        
        return observation, info

    def step(self, action):
        total_reward = 0
        
        for _ in range(self._skip_frames):
            observation, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        
        return observation, total_reward, terminated, truncated, info
class SafeDrivingWrapper(gym.Wrapper): #https://medium.com/@gomes.fs/ai-car-racing-teach-ai-to-drive-a2049ee07297
    def __init__(self, env, border_width=2):
        super(SafeDrivingWrapper, self).__init__(env)
        self.border_width = border_width
    def car_on_track(self):
        car_on_track = False
        x, y = self.unwrapped.car.hull.position
        point = Point(x, y)
        for poly in self.unwrapped.road_poly:
            polygon = Polygon(poly[0])

            if self.border_width > 0:
                border_scale = 1 + self.border_width
                polygon = affinity.scale(polygon, xfact=border_scale, yfact=border_scale)
            if polygon.contains(point):
                car_on_track = True
                break
        return car_on_track

    def step(self, action):
        next_state, reward, terminated, truncated, info = self.env.step(action)

        if not self.car_on_track():
            reward -= 300
            terminated = True 

        return next_state, reward, terminated, truncated, info
    
class RewardSpeed(gym.Wrapper):
    def get_car_speed(self):
        vel = self.env.unwrapped.car.hull.linearVelocity
        return np.sqrt(vel[0]**2 + vel[1]**2)
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info['speed'] = self.get_car_speed()

        return obs, reward, terminated, truncated, info
class RewardProgress(gym.Wrapper):
    def get_car_tiles(self):
        return self.env.unwrapped.tile_visited_count
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info['tiles'] = self.get_car_tiles()
        return obs, reward, terminated, truncated, info
    
import numpy as np
from scipy.spatial import distance

class TrackCenterWrapper(gym.Wrapper):
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        car_pos = self.env.unwrapped.car.hull.position # (x, y)
        
        track_points = np.array([(p[2], p[3]) for p in self.env.unwrapped.track])
        
        dist_to_center = np.min(np.linalg.norm(track_points - car_pos, axis=1))
        
        max_dist = 5.0 
        centering_reward = max(0, 1.0 - (dist_to_center / max_dist))
        
        total_reward = reward + (centering_reward * 2.0) 
        
        info['dist_to_center'] = dist_to_center
        
        return obs, total_reward, terminated, truncated, info

# env = GrayscaleObservation(env, keep_dim=False)
env = CarEnvironment(env)
env = RecordVideo(
    env, 
    video_folder="track-long", 
    episode_trigger=lambda x: x % training_period == 0,
    disable_logger=True 
)
env = RewardSpeed(env)
env = SafeDrivingWrapper(env)
env = TrackCenterWrapper(env)
env = RewardProgress(env)

env = GrayscaleObservation(env, keep_dim=False) 
env = ResizeObservation(env, (84, 84))           
# env = TransformObservation(env, lambda obs: np.expand_dims(obs, axis=0), 
#                            observation_space=gym.spaces.Box(low=0, high=255, shape=(1, 84, 84), dtype=np.uint8))
env = FrameStack(env, 4)
car = Car(state_dim=env.observation_space.shape, input_dim=env.action_space.n, output_dim=env.action_space.n)
all_rewards = []
speed = []
progress = 0
STALL_THRESHOLD = 50
state, info = env.reset()
for episode_num in range(num_training_episodes):
    state, info = env.reset()
    episode_over = False
    episode_reward = 0
    episode_steps = 0
    episode_speeds = []
    slow_steps = 0
    tiles = 0
    episode_max_progress = 0
    
    current_loss = 0.0
    current_q = 0.0

    while not episode_over:
        current_speed = info.get('speed', 0)
        action = car.act(state, current_speed)
        
        next_state, reward, terminated, truncated, info = env.step(action)
        tiles2 = env.unwrapped.tile_visited_count
        delta_tiles = tiles2 - tiles
        tiles = tiles2 if tiles2 > tiles else tiles
        episode_speeds.append(current_speed)
        
        if info.get('speed', 0) < 40:
            slow_steps += 1
            reward += delta_tiles * 50
            
        if slow_steps % 200 == 0:
            reward -= 300
        if slow_steps > 750:
            terminated = True
            
        progress_reward = env.unwrapped.tile_visited_count / len(env.unwrapped.track)
        if progress_reward > episode_max_progress:
            episode_max_progress = progress_reward
            
        reward = reward + (progress_reward * 10)
        reward = np.float32(reward)
        if reward <= -50:
            terminated = True
        done = terminated or truncated
        
        car.cache(state, next_state, action, reward, done)
        learn_result = car.learn() 
        
        if learn_result is not None and learn_result[0] is not None:
            current_q, current_loss = learn_result

        state = next_state
        episode_reward += reward
        episode_steps += 1
        
        pbar.set_postfix({
            'max_progress': f'{(progress*100):.2f}%',
            "Loss": f"{current_loss:.4f}",       
            "Avg_Q": f"{current_q:.2f}",
            "Step": car.curr_step,
            "Reward": f"{episode_reward:.1f}",
            "Epsilon": f"{car.exploration_rate:.2f}",
            "Speed": f"{info.get('speed', 0):.1f}",
        })
        
        if done or info.get("flag_get", False):
            all_rewards.append(episode_reward)
            break
        episode_over = done
        
    if episode_max_progress > progress:
        progress = episode_max_progress
        episodes_since_improvement = 0 
        if progress > 0.10: 
            pbar.write(f" New Progress Record: {progress*100:.2f}%! Saving model...")
            car.save(suffix=f"best_progress_{int(progress*100)}")
            car.save(suffix="best")
    else:
        episodes_since_improvement += 1
        
    if episodes_since_improvement >= STALL_THRESHOLD:
        pbar.write(f"Stall in training: {STALL_THRESHOLD} episodes without tracking progress improvement.")
        # did_load = car.load(suffix="best")
        # if did_load:
        #     if car.exploration_rate < 0.5:
        #         car.exploration_rate = max(0.50, car.exploration_rate)
        #         pbar.write(f" Exploration forced to {car.exploration_rate:.2f}")
        # else:
        #     car.exploration_rate = max(car.exploration_rate, 0.50)
        #     pbar.write("No save file found yet. Manually bumping exploration")
        episodes_since_improvement = 0
        
    if "episode" in info:
        episode_data = info["episode"]
        logging.info(f"Episode {episode_num}: "
                    f"reward={episode_data['r']:.1f}, "
                    f"length={episode_data['l']}, "
                    f"time={episode_data['t']:.2f}s")

    avg_speed = sum(episode_speeds) / len(episode_speeds) if episode_speeds else 0
    if episode_num % training_period == 0:
        recent_rewards = all_rewards[:-training_period:-1]
        avg_recent = sum(recent_rewards) / len(recent_rewards)
        print(f"  -> Average reward over last {len(recent_rewards)} episodes: {avg_recent:.1f}, speed: {avg_speed}")
    pbar.update(1)
    if episode_num == num_training_episodes -1:
        car.save()
    
pbar.close()
env.close()


Training car:   0%|          | 0/2000 [00:00<?, ?it/s]/Users/iamgeorgerieh/Documents/coding/.venv/lib/python3.10/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /Users/iamgeorgerieh/Documents/coding/track-long folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
Training car:   0%|          | 1/2000 [00:02<1:08:04,  2.04s/it, max_progress=4.71%, Loss=0.0000, Avg_Q=0.00, Step=75, Reward=108.5, Epsilon=1.00, Speed=3.4] 

  -> Average reward over last 1 episodes: 525.1, speed: 26.904539796933598


Training car:   1%|▏         | 26/2000 [00:34<50:13,  1.53s/it, max_progress=9.32%, Loss=11.5243, Avg_Q=3.04, Step=1831, Reward=177.2, Epsilon=0.92, Speed=17.6] 

  -> Average reward over last 24 episodes: 691.6, speed: 26.288777995668326


Training car:   2%|▏         | 37/2000 [00:50<1:00:09,  1.84s/it, max_progress=17.20%, Loss=12.4278, Avg_Q=6.08, Step=2675, Reward=170.5, Epsilon=0.86, Speed=13.2]

...

  -> Average reward over last 24 episodes: 5188.4, speed: 38.221052815703054


Training car:  21%|██▏       | 426/2000 [43:50<21:43:16, 49.68s/it, max_progress=100.00%, Loss=14.6223, Avg_Q=457.24, Step=84623, Reward=169.1, Epsilon=0.11, Speed=13.2]  

  -> Average reward over last 24 episodes: 6200.5, speed: 44.97419489207823


Training car:  21%|██▏       | 427/2000 [44:19<17:15:31, 39.50s/it, max_progress=100.00%, Loss=16.1479, Avg_Q=457.58, Step=85507, Reward=3638.4, Epsilon=0.11, Speed=46.6]

KeyboardInterrupt: 